In [24]:
# @title 1. Install Library
!pip install -qU langchain-groq langchain-core langgraph streamlit pyngrok requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [26]:
# @title 2. Konfigurasi API Key
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ GROQ API Key berhasil dimuat!")
except Exception as e:
    print("❌ Gagal memuat API Key. Pastikan nama secret benar (GROQ_API_KEY).")

✅ GROQ API Key berhasil dimuat!


In [31]:
# @title 3. Definisi Tools & Agent (Updated Model)
import requests
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- 1. Definisi Tools (Sama seperti sebelumnya) ---

@tool
def check_vehicle_recalls(make: str, model: str, year: str) -> str:
    """Mengecek recall kendaraan dari API NHTSA."""
    url = f"https://api.nhtsa.gov/recalls/recallsByVehicle?make={make}&model={model}&modelYear={year}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('results'):
                recalls = data['results']
                output = f"⚠️ Ditemukan {len(recalls)} recall untuk {make} {model} {year}:\n\n"
                for r in recalls[:3]:
                    output += f"- Komponen: {r.get('Component', 'N/A')}\n"
                    output += f"  Masalah: {r.get('Summary', 'N/A')}\n\n"
                return output
            else:
                return f"✅ Tidak ada recall aktif untuk {make} {model} {year}."
        else:
            return "⚠️ Gagal mengambil data dari API."
    except Exception as e:
        return f"❌ Error: {str(e)}"

@tool
def decode_error_code(code: str) -> str:
    """Menerjemahkan kode error OBD-II."""
    error_db = {
        "P0300": {"desc": "Random/Multiple Cylinder Misfire", "cause": "Busi rusak, coil pack.", "action": "Cek sistem pengapian."},
        "P0420": {"desc": "Catalyst System Efficiency Below Threshold", "cause": "Catalytic converter atau sensor O2.", "action": "Cek kondisi catalytic converter."},
        "P0171": {"desc": "System Too Lean (Bank 1)", "cause": "Vacuum leak, fuel pump lemah.", "action": "Cek kebocoran selang intake."},
        "P0442": {"desc": "Evaporative Emission Control Leak", "cause": "Tutup tangki bensin longgar.", "action": "Kencangkan tutup tangki."}
    }

    code = code.upper()
    if code in error_db:
        info = error_db[code]
        return f"🔴 **{code}**: {info['desc']}\nPenyebab: {info['cause']}\nTindakan: {info['action']}"
    return f"⚠️ Kode {code} tidak ditemukan di database lokal."

tools = [check_vehicle_recalls, decode_error_code]

# --- 2. Inisialisasi Agent dengan Model BARU ---

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # MODEL BARU (Menggantikan yang lama)
    temperature=0.2
)

system_prompt = """
Anda adalah 'AutoAssist', seorang mekanik virtual.
Aturan:
1. Jika user menyebut kode error (P0xxx), GUNAKAN tool 'decode_error_code'.
2. Jika user tanya recall mobil, GUNAKAN tool 'check_vehicle_recalls'.
3. Jawab dengan Bahasa Indonesia santai.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("✅ Agent berhasil dibuat dengan model 'llama-3.1-8b-instant'.")

✅ Agent berhasil dibuat dengan model 'llama-3.1-8b-instant'.


In [32]:
# @title 4. Testing Agent
def chat_with_agent(user_input: str):
    try:
        response = agent.invoke({"messages": [("user", user_input)]})
        return response["messages"][-1].content
    except Exception as e:
        return f"Error: {str(e)}"

print("User: Lampu check engine menyala kode P0171, itu apa ya?")
print("Bot:", chat_with_agent("Lampu check engine menyala kode P0171, itu apa ya?"))

User: Lampu check engine menyala kode P0171, itu apa ya?
Bot: Jadi, kode P0171 menunjukkan bahwa sistem bahan bakar mobil Anda terlalu kering. Ini bisa disebabkan oleh kebocoran selang intake atau fuel pump yang lemah. Saya sarankan Anda untuk memeriksa kebocoran selang intake dan memeriksa kondisi fuel pump.


In [33]:
# @title 5. Buat File Aplikasi Streamlit
%%writefile streamlit_app.py

import streamlit as st
import requests
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- KONFIGURASI HALAMAN ---
st.set_page_config(page_title="AutoAssist Bot", page_icon="🚗")

# --- 1. DEFINISI TOOLS ---

@tool
def check_vehicle_recalls(make: str, model: str, year: str) -> str:
    """Mengecek recall kendaraan dari API NHTSA."""
    url = f"https://api.nhtsa.gov/recalls/recallsByVehicle?make={make}&model={model}&modelYear={year}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('results'):
                recalls = data['results']
                output = f"⚠️ Ditemukan {len(recalls)} recall untuk {make} {model} {year}:\n\n"
                for r in recalls[:3]:
                    output += f"- Komponen: {r.get('Component', 'N/A')}\n"
                    output += f"  Masalah: {r.get('Summary', 'N/A')}\n\n"
                return output
            else:
                return f"✅ Tidak ada recall aktif untuk {make} {model} {year}."
        else:
            return "⚠️ Gagal mengambil data API."
    except Exception as e:
        return f"❌ Error: {str(e)}"

@tool
def decode_error_code(code: str) -> str:
    """Menerjemahkan kode error OBD-II."""
    error_db = {
        "P0300": {"desc": "Random/Multiple Cylinder Misfire", "cause": "Busi rusak, coil pack.", "action": "Cek sistem pengapian."},
        "P0420": {"desc": "Catalyst System Efficiency Below Threshold", "cause": "Catalytic converter atau sensor O2.", "action": "Cek kondisi catalytic converter."},
        "P0171": {"desc": "System Too Lean (Bank 1)", "cause": "Vacuum leak, fuel pump lemah.", "action": "Cek kebocoran selang intake."},
        "P0442": {"desc": "Evaporative Emission Control Leak", "cause": "Tutup tangki bensin longgar.", "action": "Kencangkan tutup tangki."}
    }

    code = code.upper()
    if code in error_db:
        info = error_db[code]
        return f"🔴 **{code}**: {info['desc']}\nPenyebab: {info['cause']}\nTindakan: {info['action']}"
    return f"⚠️ Kode {code} tidak ditemukan di database lokal."

tools = [check_vehicle_recalls, decode_error_code]

# --- 2. SIDEBAR PENGATURAN ---
with st.sidebar:
    st.title("⚙️ Pengaturan")
    groq_api_key = st.text_input("Masukkan Groq API Key", type="password", help="Dapatkan key gratis di console.groq.com")

    st.markdown("---")
    st.markdown("### 🚗 Tentang AutoAssist")
    st.markdown("Bot konsultasi mobil cerdas menggunakan **Llama 3.1** via Groq.")
    st.markdown("Dibuat untuk memenuhi tugas Final Project LLM.")

    if st.button("Reset Chat"):
        st.session_state.messages = []
        st.rerun()

# --- 3. VALIDASI API KEY ---
if not groq_api_key:
    st.info("👋 Silakan masukkan Groq API Key di sidebar untuk memulai.", icon="🔐")
    st.stop()

# --- 4. INISIALISASI SESSION STATE ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- 5. INISIALISASI AGENT ---
# Menggunakan cache agar tidak reload terus menerus
@st.cache_resource
def get_agent(api_key):
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.2,
        groq_api_key=api_key
    )
    system_prompt = """
    Anda adalah 'AutoAssist', mekanik virtual.
    Aturan:
    1. Jika user menyebut kode error (P0xxx), GUNAKAN tool 'decode_error_code'.
    2. Jika user tanya recall mobil, GUNAKAN tool 'check_vehicle_recalls'.
    3. Jawab dengan Bahasa Indonesia santai.
    """
    return create_agent(model=llm, tools=tools, system_prompt=system_prompt)

try:
    agent = get_agent(groq_api_key)
except Exception as e:
    st.error(f"Gagal memuat model: {e}")
    st.stop()

# --- 6. TAMPILAN CHAT ---
st.title("🚗 AutoAssist Bot")
st.caption("Asisten mekanik virtual Anda. Tanyakan masalah mobil Anda di sini.")

# Tampilkan history chat
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Logic Input Chat
if prompt := st.chat_input("Masukkan pertanyaan mobil Anda..."):
    # 1. Tambah pesan user
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # 2. Panggil Agent
    with st.chat_message("assistant"):
        with st.spinner("Menganalisis..."):
            try:
                response = agent.invoke({"messages": [("user", prompt)]})
                answer = response["messages"][-1].content
                st.markdown(answer)
                st.session_state.messages.append({"role": "assistant", "content": answer})
            except Exception as e:
                error_msg = f"Maaf, terjadi error: {str(e)}"
                st.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})

print("✅ File streamlit_app.py berhasil dibuat!")

Writing streamlit_app.py


In [35]:
# @title 6. Jalankan Streamlit dengan Authtoken
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata

# --- KONFIGURASI NGROK ---
try:
    # Ambil token dari secrets
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)
    print("✅ Ngrok Auth Token berhasil dimuat.")
except Exception as e:
    print("❌ Gagal memuat NGROK_TOKEN. Pastikan sudah disimpan di Colab Secrets.")
    print("Error:", e)

# --- MULAI APLIKASI ---

# Hapus proses lama jika ada
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["fuser", "-k", "8501/tcp"], capture_output=True)
ngrok.kill()

# Jalankan Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py", "--server.headless=true", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Tunggu server siap
time.sleep(5)

# Buka tunnel ngrok
try:
    public_url = ngrok.connect(8501)
    print(f"🚀 Aplikasi Anda sudah ONLINE! Klik link di bawah ini:")
    print(public_url)
    print("\n💡 CATATAN: Masukkan Groq API Key di sidebar aplikasi.")
except Exception as e:
    print(f"❌ Error membuat tunnel: {e}")

✅ Ngrok Auth Token berhasil dimuat.
🚀 Aplikasi Anda sudah ONLINE! Klik link di bawah ini:
NgrokTunnel: "https://immovable-accustom-resubmit.ngrok-free.dev" -> "http://localhost:8501"

💡 CATATAN: Masukkan Groq API Key di sidebar aplikasi.
